# 01. Pooled embeddings: ESM C 600M and ProtT5-XL

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

In [ ]:
import os
from glob import glob

DRIVE_ROOT = "/content/drive/MyDrive/Diplomska_v2"
if not os.path.isdir(DRIVE_ROOT):
    shared = glob("/content/drive/.shortcut-targets-by-id/*/Diplomska_v2")
    if shared:
        DRIVE_ROOT = shared[0]
assert os.path.isdir(DRIVE_ROOT), f"Folder not found: {DRIVE_ROOT}"

PDB_SET = "kinase_domain"

os.environ["DTA_DATA_ROOT"] = DRIVE_ROOT
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("DTA_DATA_ROOT:", os.environ["DTA_DATA_ROOT"])
print("Structure set:", PDB_SET)

DTA_DATA_ROOT: /content/drive/MyDrive/Diplomska_v2
Structure set: kinase_domain


## 3. GPU

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv

Torch version: 2.11.0+cu130
CUDA available: True
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 4. Install packages

In [ ]:
!pip install -q torch_geometric biopython esm transformers sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.

## 5. Go to the folder

In [ ]:
%cd {DRIVE_ROOT}
!pwd

/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2
/content/drive/.shortcut-targets-by-id/1N088IPJAiucR3uoU-A7ezJZQCcXNDgzY/Diplomska_v2


## 6. Helpers

In [ ]:
import re, subprocess

QUIET_PREFIXES = ("Train epoch:", "Training on ", "Predicting for validation data",
                  "Make prediction for", "MSE improved at epoch", "No improvement since epoch")
VALIDATION_LINE = re.compile(r"^Epoch (\d+) - Validation MSE")

def run(args, stop_marker=None, every=10):
    cmd = ["python", "-u"] + args
    print("Running:", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stopped = False
    try:
        for line in proc.stdout:
            match = VALIDATION_LINE.match(line)
            if match:
                if int(match.group(1)) % every == 0:
                    print(line, end="")
                continue
            if line.startswith(QUIET_PREFIXES):
                continue
            print(line, end="")
            if stop_marker and stop_marker in line:
                stopped = True
                print("\nStopping here as requested. Rerun this cell to continue; finished folds are skipped.")
                break
    finally:
        if proc.poll() is None:
            proc.terminate()
        proc.wait()
    if not stopped and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}, see the output above.")

## 7. Check helper

In [ ]:
import json
import torch
from Paths import paths_for
from Protein_Embeddings import build_adapter, load_pooled_embeddings

PATHS = paths_for("davis", PDB_SET)

def check_embeddings(model_name):
    expected_dim = build_adapter(model_name).embedding_dim
    pooled = load_pooled_embeddings(PATHS.protein_embeddings, model_name, PATHS.pdb_file)
    with open(os.path.join(PATHS.protein_embeddings, "sequences_pdb.json")) as f:
        sequences = json.load(f)["sequences"]
    stacked = torch.stack(list(pooled.values()))
    unique_vectors = torch.unique(stacked, dim=0).shape[0]
    unique_sequences = len(set(sequences.values()))
    constant_dims = int((stacked.std(dim=0) == 0).sum())
    print(f"model: {model_name}")
    print(f"proteins: {len(pooled)} (expected 442)")
    print(f"dimension: {stacked.shape[1]} (expected {expected_dim})")
    print(f"all finite: {bool(torch.isfinite(stacked).all())}")
    print(f"distinct vectors: {unique_vectors}, unique sequences: {unique_sequences}")
    print(f"constant dimensions: {constant_dims}")
    print(f"mean vector norm: {stacked.norm(dim=1).mean():.2f}, min {stacked.min():.2f}, max {stacked.max():.2f}")
    with open(os.path.join(PATHS.protein_embeddings, model_name, "windowed_pdb.json")) as f:
        print("windowed:", json.load(f))
    ok = (len(pooled) == 442 and stacked.shape[1] == expected_dim and bool(torch.isfinite(stacked).all())
          and unique_vectors == unique_sequences and constant_dims == 0)
    print("OK" if ok else "PROBLEM: these embeddings failed the checks")

# ESM C 600M

In [ ]:
run(["compute_protein_embeddings.py", "--model", "esmc_600m", "--pdb-set", PDB_SET])

Running: python -u compute_protein_embeddings.py --model esmc_600m --pdb-set kinase_domain
Structures: kinase_domain (/content/drive/MyDrive/Diplomska_v2/data/davis/pdb_kinase_domain)
442 proteins, 428 unique sequences, 428 to embed with esmc_600m
Loading esmc_600m on cuda
ESMC: Transformer Engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically - measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack this reaches ~O(100) m

In [ ]:
check_embeddings("esmc_600m")

model: esmc_600m
proteins: 442 (expected 442)
dimension: 1152 (expected 1152)
all finite: True
distinct vectors: 428, unique sequences: 428
constant dimensions: 0
mean vector norm: 0.89, min -0.69, max 0.27
windowed: {'lengths': {}, 'max_residues': 2046}
OK


# ProtT5-XL
The code feeds ProtT5 the residues separated by spaces, maps the rare amino acids U, Z, O, B to X, uses the encoder only and drops the end token before averaging over residues. This is the recommended input for the Rostlab models.

In [ ]:
run(["compute_protein_embeddings.py", "--model", "prot_t5_xl_uniref50", "--pdb-set", PDB_SET])

Running: python -u compute_protein_embeddings.py --model prot_t5_xl_uniref50 --pdb-set kinase_domain
Structures: kinase_domain (/content/drive/MyDrive/Diplomska_v2/data/davis/pdb_kinase_domain)
442 proteins, 428 unique sequences, 428 to embed with prot_t5_xl_uniref50
Loading prot_t5_xl_uniref50 on cuda
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
I0000 00:00:1791413870.809152    3191 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other

In [ ]:
check_embeddings("prot_t5_xl_uniref50")

model: prot_t5_xl_uniref50
proteins: 442 (expected 442)
dimension: 1024 (expected 1024)
all finite: True
distinct vectors: 428, unique sequences: 428
constant dimensions: 0
mean vector norm: 1.25, min -0.26, max 0.27
windowed: {'lengths': {}, 'max_residues': None}
OK
